# Lambda and Useful Built-in Functions

**Session 06 (2026-10-24) · Priority 2 · Notebook 6 of 6**

Python comes with built-in functions that replace a lot of hand-written loops. This notebook covers `lambda` first, because most of the others take a function as an argument, and then `enumerate()`, `zip()`, `map()`, `filter()`, `any()`, `all()` and `sorted()`.

Each one is explained the same way:

- **What it is**: the idea in one or two sentences
- **Behind the scenes**: what Python actually does
- **Syntax**: how to call it
- **Returns**: what you get back
- **How to use**: examples, gotchas and real-world uses

---
## 1. `lambda`

**What it is:** a small, **anonymous** (nameless) function written in one line. It's handy when you need a short function just once, usually as an argument to another function.

**Behind the scenes:** `lambda` creates an ordinary function object, exactly like `def`. The only differences are that it has no name, and its body is a **single expression** whose value is returned automatically (there's no `return` keyword).

**Syntax:** `lambda parameters: expression`
**Returns:** a function. Calling that function returns the value of the expression.

In [ ]:
def square(x):
    return x * x

square_lambda = lambda x: x * x       # the same function, written as a lambda

print(square(5))
print(square_lambda(5))
print(type(square), type(square_lambda))   # both are functions

A lambda can take several parameters, have default values, and use a conditional expression:

In [ ]:
add = lambda a, b: a + b
greet = lambda name, msg="hello": f"{msg}, {name}!"
grade = lambda mark: "pass" if mark >= 50 else "fail"

print(add(2, 3))
print(greet("asha"))
print(grade(72), grade(30))

| A lambda can | A lambda can't |
|---|---|
| take any number of parameters, including defaults | contain statements: `if` blocks, loops, `return`, `=` assignments, `try` |
| use a conditional expression: `a if cond else b` | hold more than one expression |

**Style note:** giving a lambda a name, as in `square = lambda x: x * x`, is discouraged in real code (PEP 8, Python's style guide). If it needs a name, use `def`. Lambdas shine when passed **directly** to another function, which is what the rest of this notebook does. A quick preview:

In [ ]:
students = [("asha", 92), ("ravi", 45), ("monal", 78)]

# sort by the mark: for each student tuple s, sort by s[1]
print(sorted(students, key=lambda s: s[1]))

---
## 2. `enumerate()`

**What it is:** gives you **the index and the item together** while looping, so you don't need a separate counter variable.

**Behind the scenes:** it's a **lazy iterator**. It doesn't build a list in advance. Each time the loop asks for the next value, it returns a tuple `(count, item)` and adds 1 to its count.

**Syntax:** `enumerate(iterable, start=0)`
**Returns:** an `enumerate` object that produces `(index, item)` tuples.

In [ ]:
topics = ["exceptions", "os", "files", "oop"]

# Without enumerate: manage a counter yourself
i = 0
for topic in topics:
    print(i, topic)
    i += 1

In [ ]:
# With enumerate
for index, topic in enumerate(topics):
    print(index, topic)

print()
for number, topic in enumerate(topics, start=1):   # start counting at 1
    print(f"{number}. {topic}")

To see what it produces, convert it to a list:

In [ ]:
e = enumerate(topics)
print(e)              # an enumerate object, not a list
print(list(e))        # the (index, item) tuples

**Use case:** find the positions of items that match a condition.

In [ ]:
marks = [92, 45, 78, 30, 66]
failed_positions = [i for i, m in enumerate(marks) if m < 50]
print(failed_positions)

---
## 3. `zip()`

**What it is:** walks through several iterables **side by side**, pairing up their items: first with first, second with second, and so on.

**Behind the scenes:** a lazy iterator. At each step it takes the next item from **every** iterable and returns them together as a tuple. It **stops at the shortest** iterable, without any warning.

**Syntax:** `zip(iterable_1, iterable_2, ..., strict=False)`
**Returns:** a `zip` object that produces tuples.

In [ ]:
names = ["asha", "ravi", "monal"]
marks = [92, 45, 78]

for name, mark in zip(names, marks):
    print(name, mark)

print(list(zip(names, marks)))

Pairs convert straight into a dictionary:

In [ ]:
print(dict(zip(names, marks)))

**Gotcha:** different lengths are silently cut to the shortest. Pass `strict=True` (Python 3.10+) to get an error instead:

In [ ]:
numbers = [1, 2]
letters = ["a", "b", "c", "d"]
print(list(zip(numbers, letters)))       # 'c' and 'd' are silently dropped

try:
    list(zip(numbers, letters, strict=True))
except ValueError as e:
    print("ValueError:", e)

It works with any number of iterables:

In [ ]:
cities = ["Pune", "Delhi", "Chennai"]
for name, mark, city in zip(names, marks, cities):
    print(f"{name} ({city}): {mark}")

**Unzipping:** `zip(*pairs)` splits pairs back into separate sequences. The `*` spreads the list out into separate arguments, as with `*args`.

In [ ]:
pairs = [("asha", 92), ("ravi", 45), ("monal", 78)]
names_back, marks_back = zip(*pairs)     # same as zip(("asha", 92), ("ravi", 45), ("monal", 78))
print(names_back)
print(marks_back)

---
## 4. `map()`

**What it is:** applies a function to **every item** of an iterable.

**Behind the scenes:** `map()` is **lazy**. Calling it computes nothing. Each value is computed only when something asks for it (a loop, `list()`, `sum()`). Once all the values have been taken, the map object is **empty**: iterators can only be used once.

**Syntax:** `map(function, iterable, ...)`
**Returns:** a `map` object (an iterator). Wrap it in `list()` to see every value.

In [ ]:
values = ["1", "2", "3", "4"]

numbers = map(int, values)        # pass the function itself: int, not int()
print(numbers)                    # a map object: nothing has been computed yet
print(list(numbers))              # now the values are computed
print(list(numbers))              # [] : it has already been used up!

Any function works: a lambda, a built-in, or a method such as `str.title`:

In [ ]:
prices = [100, 250, 80]
with_tax = list(map(lambda p: round(p * 1.18, 2), prices))
print(with_tax)

names = ["asha", "ravi"]
print(list(map(str.title, names)))

With several iterables, the function receives one item from each:

In [ ]:
a = [1, 2, 3]
b = [10, 20, 30]
print(list(map(lambda x, y: x + y, a, b)))

**`map` or a comprehension?** They give the same result:

In [ ]:
print(list(map(lambda p: p * 2, prices)))
print([p * 2 for p in prices])

**Rule of thumb:** if you already have a named function (`int`, `str.upper`), `map` is neat. If you'd need to write a lambda, a comprehension is usually easier to read.

---
## 5. `filter()`

**What it is:** keeps only the items for which a function returns `True` (or any truthy value).

**Behind the scenes:** a lazy iterator, like `map`. It calls the function on each item and passes the item on only if the result is truthy. If you pass `None` as the function, it keeps the items that are truthy themselves.

**Syntax:** `filter(function, iterable)`
**Returns:** a `filter` object (an iterator).

In [ ]:
marks = [92, 45, 78, 30, 66]

passed = filter(lambda m: m >= 50, marks)
print(passed)
print(list(passed))

With a regular named function:

In [ ]:
def is_valid_email(email):
    return "@" in email and email.endswith(".com")

emails = ["asha@gmail.com", "not-an-email", "ravi@yahoo.com", "monal@site"]
print(list(filter(is_valid_email, emails)))

With `None`, it removes every falsy value (`""`, `None`, `0`, `[]`, ...):

In [ ]:
messy = ["asha", "", None, "ravi", 0, "monal", []]
print(list(filter(None, messy)))

The comprehension equivalent of `filter(lambda m: m >= 50, marks)` is `[m for m in marks if m >= 50]`.

---
## 6. `any()` and `all()`

**What they are:**

- `any(iterable)` returns `True` if **at least one** item is truthy.
- `all(iterable)` returns `True` if **every** item is truthy.

**Behind the scenes:** both **short-circuit**. `any()` stops at the first truthy item and `all()` stops at the first falsy item; the rest are never checked. For an empty iterable, `any([])` is `False` (nothing is true) and `all([])` is `True` (nothing is false).

**Syntax:** `any(iterable)`, `all(iterable)`
**Returns:** `True` or `False`.

| Input | `any()` | `all()` |
|---|---|---|
| `[True, False]` | `True` | `False` |
| `[True, True]` | `True` | `True` |
| `[False, False]` | `False` | `False` |
| `[]` (empty) | `False` | `True` |

In [ ]:
print(any([False, True, False]))
print(all([True, True, False]))
print(any([]), all([]))

They're most useful with a **generator expression** that asks a yes/no question about every item:

In [ ]:
marks = [92, 45, 78, 30, 66]

print("did anyone fail?     ", any(m < 50 for m in marks))
print("did everyone pass?   ", all(m >= 50 for m in marks))
print("are all marks valid? ", all(0 <= m <= 100 for m in marks))

**Real-world use:** a password strength check.

In [ ]:
password = "Fde2026bootcamp"

checks = {
    "has a digit": any(ch.isdigit() for ch in password),
    "has upper case": any(ch.isupper() for ch in password),
    "has lower case": any(ch.islower() for ch in password),
    "8+ characters": len(password) >= 8,
}
print(checks)
print("strong password?", all(checks.values()))

**Short-circuiting in action:** `any()` stops as soon as it finds a `True`, so 5 and 7 are never checked.

In [ ]:
def check(n):
    print(f"  checking {n}")
    return n > 2

print(any(check(n) for n in [1, 3, 5, 7]))

---
## 7. `sorted()`

**What it is:** returns a **new sorted list** from any iterable. The original is left unchanged.

**Behind the scenes:** Python's sorting algorithm (Timsort) compares items with `<`. With `key=`, it first calls the key function **once per item** and sorts by those results instead. The sort is **stable**: items with equal keys keep their original order.

**Syntax:** `sorted(iterable, key=None, reverse=False)`
**Returns:** a new `list`.

| | `sorted(x)` | `x.sort()` |
|---|---|---|
| Works on | any iterable: list, tuple, string, set, dict | lists only |
| The original | unchanged | changed in place |
| Returns | a new list | `None` |

In [ ]:
marks = [78, 92, 45, 66]

new_list = sorted(marks)
print(new_list)
print(marks)                    # unchanged

result = marks.sort()           # sorts the list in place
print(marks)
print(result)                   # None: never write marks = marks.sort()!

`sorted()` accepts any iterable and always returns a list:

In [ ]:
print(sorted("python"))                  # a string becomes a sorted list of characters
print(sorted({3, 1, 2}))                 # a set
print(sorted({"b": 2, "a": 1}))          # a dict: its keys, sorted
print(sorted(marks, reverse=True))       # largest first

**`key=`** tells `sorted` *what* to sort by. You pass a function, without calling it:

In [ ]:
names = ["Ravi", "asha", "Monal", "alex"]

print(sorted(names))                     # capitals first: 'R' comes before 'a'
print(sorted(names, key=str.lower))      # case-insensitive
print(sorted(names, key=len))            # by length; ties keep their original order

**Sorting records** with a lambda. Return a **tuple** from the key to sort by several things: first the first value, then the second for ties.

In [ ]:
students = [("ravi", 45, "ECE"), ("monal", 78, "CSE"), ("asha", 92, "CSE"), ("alex", 78, "ECE"), ("srinu", 66, "CSE")]

by_mark = sorted(students, key=lambda s: s[1], reverse=True)
print(by_mark)            # monal and alex tie on 78 and keep their original order (stable sort)

by_dept_then_name = sorted(students, key=lambda s: (s[2], s[0]))
print(by_dept_then_name)

**Sorting a dict by its values:**

In [ ]:
prices = {"apple": 120, "banana": 40, "mango": 150}

print(sorted(prices.items(), key=lambda item: item[1]))      # list of (key, value) pairs
cheapest_first = dict(sorted(prices.items(), key=lambda item: item[1]))
print(cheapest_first)

Items that can't be compared with `<` raise a `TypeError`:

In [ ]:
try:
    sorted([3, "a", 1])
except TypeError as e:
    print("TypeError:", e)

---
## 8. Putting it together

A small leaderboard from raw text data, using `zip`, a comprehension, `sorted` with a lambda, `enumerate`, `all` and `map`. (`_` is a conventional name for a value you don't need.)

In [ ]:
names = ["asha", "ravi", "monal", "alex", "srinu"]
raw_marks = ["92", "45", "78", "abc", "66"]        # raw text with one bad value

# 1. pair names with marks, keeping only the valid numbers
valid = [(n, int(m)) for n, m in zip(names, raw_marks) if m.isdigit()]

# 2. rank by mark, highest first
ranked = sorted(valid, key=lambda pair: pair[1], reverse=True)

# 3. print the leaderboard: each item is (rank, (name, mark))
for rank, (name, mark) in enumerate(ranked, start=1):
    status = "pass" if mark >= 50 else "fail"
    print(f"{rank}. {name:<6} {mark:>3}  {status}")

print("everyone passed?", all(mark >= 50 for _, mark in valid))
print("class average   :", sum(map(lambda pair: pair[1], valid)) / len(valid))

---
## Cheat sheet

| Function | Returns | Lazy? | In one line |
|---|---|---|---|
| `lambda x: expr` | a function | n/a | a tiny anonymous function |
| `enumerate(it, start=0)` | iterator of `(index, item)` | Yes | the index together with the item |
| `zip(a, b, ...)` | iterator of tuples | Yes | pairs items up; stops at the shortest |
| `map(f, it)` | iterator | Yes | applies `f` to every item |
| `filter(f, it)` | iterator | Yes | keeps items where `f` is truthy |
| `any(it)` | `bool` | short-circuits | at least one item is truthy |
| `all(it)` | `bool` | short-circuits | every item is truthy |
| `sorted(it, key=None, reverse=False)` | a new list | No | sorts anything |

**Lazy** means the values are produced one at a time, only when needed, and the iterator can be used only once. Wrap it in `list()` to see or keep the values.

## Practice

**1.** Use `enumerate` to print a numbered menu from `["Tea", "Coffee", "Juice"]`, starting at 1.

In [ ]:
# Exercise 1

**2.** Use `zip` to combine `products = ["pen", "book", "bag"]` and `prices = [10, 250, 800]` into a dict.

In [ ]:
# Exercise 2

**3.** Use `map` to convert `["10", "20", "30"]` to integers, then `sum` them.

In [ ]:
# Exercise 3

**4.** Use `filter` with a lambda to keep the words longer than 4 characters from `["data", "python", "ai", "engineer"]`.

In [ ]:
# Exercise 4

**5.** Using `any` and `all`, check whether a list of ages contains any minors (under 18), and whether everyone is under 60.

In [ ]:
# Exercise 5

**6.** Sort `employees = [("asha", 50000), ("ravi", 72000), ("monal", 61000)]` by salary, highest first, and print the top earner's name.

In [ ]:
# Exercise 6